# MBDD2025 training walkthrough (Google Colab)

This notebook follows the project training pipeline in visible stages: inspect the source files, validate YOLO labels, create the seeded train/validation split, configure and train YOLO26s, evaluate the saved checkpoint, and run a prediction demo.

**Important:** training uses `Labels/*.txt` in YOLO detection format. `Annotations/*.xml` is only counted/inspected here; the project training script does not convert VOC XML. If `Labels` does not contain matching YOLO `.txt` files, stop and convert the XML annotations first.

In Colab, select a GPU runtime before running the cells. The notebook uses `device=0`, which is the single GPU assigned to the Colab runtime. Outputs and checkpoints are written to Google Drive; the prepared split is built under `/content` for the lifetime of the runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DATASET_PATH = "/content/drive/MyDrive/......" # Use the path to your dataset here
# Verify that the dataset directories for images annotations and labels exist. If not, raise an error to inform the user.:
IMAGE_DIR = os.path.join(DATASET_PATH, "JPEGImages")
ANNOT_DIR = os.path.join(DATASET_PATH, "Annotations")
LABELS_DIR = os.path.join(DATASET_PATH, "Labels")

# Keep run outputs outside the source dataset directory.
OUTPUT_ROOT = Path("/content/drive/MyDrive/.......") # Use the path to your output directory here
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREPARED_DIR = Path("/content/mbdd2025_prepared_seed17") # Use the path to your prepared dataset directory here
DATASET_YAML = PREPARED_DIR / "dataset.yaml"

for folder in (IMAGE_DIR, ANNOT_DIR, LABELS_DIR):
    if not Path(folder).is_dir():
        raise FileNotFoundError(f"Expected dataset directory does not exist: {folder}")

print(f"Images:      {IMAGE_DIR}")
print(f"VOC XML:     {ANNOT_DIR}")
print(f"YOLO labels: {LABELS_DIR}")
print(f"Run output:  {OUTPUT_ROOT}")

## 1. Install the training package

Colab supplies its own PyTorch build for the selected runtime. This cell installs Ultralytics and YAML support without using the repository's CPU-only `constraints-rocm.txt` file.

In [ ]:
%pip install -q ultralytics PyYAML

import json
import math
import random
import shutil
import statistics
import time
from collections import Counter
from datetime import datetime, timezone

import torch
import yaml
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print(f"PyTorch: {torch.__version__}")
print(f"CUDA runtime reported by PyTorch: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Colab GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU is visible. In Colab, choose Runtime > Change runtime type > GPU, then rerun this cell.")

## 2. Inspect and validate the source dataset

The training code pairs each image with a same-stem `.txt` label, checks five-field normalized bounding-box rows and class IDs, and rejects missing or extra labels. An empty text file is valid and represents an image with no objects.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
CLASS_NAMES = ["crack", "leakage", "abscission", "corrosion", "bulge"]
CLASS_ID_TO_NAME = dict(enumerate(CLASS_NAMES))

image_root = Path(IMAGE_DIR)
label_root = Path(LABELS_DIR)
xml_root = Path(ANNOT_DIR)
image_paths = sorted(path for path in image_root.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
label_paths = {path.stem: path for path in label_root.glob("*.txt") if path.is_file()}
xml_paths = sorted(xml_root.glob("*.xml"))
image_stems = {path.stem for path in image_paths}
missing_labels = sorted(image_stems - label_paths.keys())
extra_labels = sorted(label_paths.keys() - image_stems)

print(f"Images found:               {len(image_paths):,}")
print(f"YOLO .txt labels found:     {len(label_paths):,}")
print(f"VOC .xml annotations found: {len(xml_paths):,}")
print(f"Class mapping: {CLASS_ID_TO_NAME}")

if not image_paths:
    raise ValueError(f"No supported images were found in {image_root}")
if missing_labels:
    raise FileNotFoundError(f"{len(missing_labels)} images have no matching YOLO .txt label; first missing stem: {missing_labels[0]}")
if extra_labels:
    raise ValueError(f"{len(extra_labels)} YOLO .txt labels have no matching image; first extra stem: {extra_labels[0]}")

class_counts = Counter()
empty_label_count = 0
object_count = 0
for image_path in image_paths:
    label_path = label_paths[image_path.stem]
    rows = [line.split() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not rows:
        empty_label_count += 1
    for row_number, fields in enumerate(rows, start=1):
        if len(fields) != 5:
            raise ValueError(f"Invalid YOLO row at {label_path}:{row_number}; expected class_id x_center y_center width height")
        try:
            class_id = int(fields[0])
            x_center, y_center, width, height = map(float, fields[1:])
        except ValueError as exc:
            raise ValueError(f"Non-numeric YOLO row at {label_path}:{row_number}") from exc
        if class_id not in CLASS_ID_TO_NAME:
            raise ValueError(f"Class ID {class_id} at {label_path}:{row_number} is outside the configured class list")
        if not all(math.isfinite(value) for value in (x_center, y_center, width, height)) or not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
            raise ValueError(f"Invalid normalized box at {label_path}:{row_number}")
        class_counts[class_id] += 1
        object_count += 1

print(f"Images with empty labels: {empty_label_count:,}")
print(f"Bounding boxes: {object_count:,}")
print("Objects per class:")
for class_id, class_name in CLASS_ID_TO_NAME.items():
    print(f"  {class_id}: {class_name:<12} {class_counts[class_id]:,}")
print("\nSample label file:")
sample_label_path = label_paths[image_paths[0].stem]
print(f"{sample_label_path.name}:")
print(sample_label_path.read_text(encoding="utf-8")[:1000] or "(empty label file)")

## 3. Visualize a source image and its labels

This quick plot lets you catch class-order or coordinate-format mistakes before spending time training.

In [ ]:
sample_image_path = image_paths[0]
sample_label_path = label_paths[sample_image_path.stem]
sample_image = Image.open(sample_image_path).convert("RGB")
image_width, image_height = sample_image.size

figure, axis = plt.subplots(figsize=(12, 8))
axis.imshow(sample_image)
axis.set_title(f"{sample_image_path.name} | {image_width} x {image_height}")
axis.axis("off")
for line in sample_label_path.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    class_id_text, x_center_text, y_center_text, width_text, height_text = line.split()
    class_id = int(class_id_text)
    x_center, y_center, box_width, box_height = map(float, (x_center_text, y_center_text, width_text, height_text))
    x_left = (x_center - box_width / 2) * image_width
    y_top = (y_center - box_height / 2) * image_height
    rectangle = patches.Rectangle((x_left, y_top), box_width * image_width, box_height * image_height, linewidth=2, edgecolor="lime", facecolor="none")
    axis.add_patch(rectangle)
    axis.text(x_left, y_top, CLASS_ID_TO_NAME[class_id], color="black", fontsize=10, bbox={"facecolor": "lime", "alpha": 0.8, "pad": 2})
plt.show()

## 4. Create the reproducible train/validation split

This matches the project preparation routine: sort image/label pairs, shuffle with seed 17, assign 20% to validation, and create `images/train2017`, `images/val2017`, `labels/train2017`, and `labels/val2017`. Symlinks avoid duplicating the Drive dataset. Set `LINK_FILES = False` if symlinks are unsuitable, keeping in mind that copying the full dataset requires substantial Colab disk space and time.

In [ ]:
SEED = 17
VALIDATION_FRACTION = 0.2
LINK_FILES = True

source_images_resolved = str(image_root.resolve())
source_labels_resolved = str(label_root.resolve())

if PREPARED_DIR.exists() and any(PREPARED_DIR.iterdir()):
    manifest_path = PREPARED_DIR / "split_manifest.json"
    if not manifest_path.is_file() or not DATASET_YAML.is_file():
        raise FileExistsError(f"Prepared output already exists but is incomplete: {PREPARED_DIR}. Change PREPARED_DIR to a fresh /content path.")
    existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    matches = (existing_manifest.get("source_images") == source_images_resolved and existing_manifest.get("source_labels") == source_labels_resolved and existing_manifest.get("seed") == SEED and existing_manifest.get("validation_fraction") == VALIDATION_FRACTION and existing_manifest.get("classes") == CLASS_NAMES)
    if not matches:
        raise FileExistsError(f"Prepared output does not match the current source/config: {PREPARED_DIR}. Change PREPARED_DIR to a fresh /content path.")
    print(f"Reusing prepared dataset: {PREPARED_DIR}")
else:
    pairs = [(image_path, label_paths[image_path.stem]) for image_path in image_paths]
    shuffled_pairs = pairs.copy()
    random.Random(SEED).shuffle(shuffled_pairs)
    validation_count = max(1, round(len(shuffled_pairs) * VALIDATION_FRACTION))
    split_pairs = {"val2017": shuffled_pairs[:validation_count], "train2017": shuffled_pairs[validation_count:]}

    for split_name, split_items in split_pairs.items():
        image_output = PREPARED_DIR / "images" / split_name
        label_output = PREPARED_DIR / "labels" / split_name
        image_output.mkdir(parents=True, exist_ok=True)
        label_output.mkdir(parents=True, exist_ok=True)
        for image_path, label_path in split_items:
            image_target = image_output / image_path.name
            label_target = label_output / label_path.name
            if LINK_FILES:
                image_target.symlink_to(image_path.resolve())
                label_target.symlink_to(label_path.resolve())
            else:
                shutil.copy2(image_path, image_target)
                shutil.copy2(label_path, label_target)

    dataset_definition = {
        "dataset_name": "MBDD2025",
        "path": str(PREPARED_DIR.resolve()),
        "train": "images/train2017",
        "val": "images/val2017",
        "names": CLASS_ID_TO_NAME,
    }
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    DATASET_YAML.write_text(yaml.safe_dump(dataset_definition, sort_keys=False), encoding="utf-8")
    split_manifest = {
        "dataset": "MBDD2025",
        "source_images": source_images_resolved,
        "source_labels": source_labels_resolved,
        "output": str(PREPARED_DIR.resolve()),
        "split_method": "seeded image-level shuffle",
        "seed": SEED,
        "validation_fraction": VALIDATION_FRACTION,
        "link_files": LINK_FILES,
        "classes": CLASS_NAMES,
        "counts": {name: len(items) for name, items in split_pairs.items()},
        "files": {name: [image.name for image, _ in items] for name, items in split_pairs.items()},
    }
    (PREPARED_DIR / "split_manifest.json").write_text(json.dumps(split_manifest, indent=2) + "\n", encoding="utf-8")
    print("Prepared split and wrote dataset.yaml plus split_manifest.json.")

split_counts = {}
for split_name in ("train2017", "val2017"):
    split_image_dir = PREPARED_DIR / "images" / split_name
    split_label_dir = PREPARED_DIR / "labels" / split_name
    split_image_paths = [path for path in split_image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS]
    missing_split_labels = [path.name for path in split_image_paths if not (split_label_dir / f"{path.stem}.txt").is_file()]
    if missing_split_labels:
        raise FileNotFoundError(f"Prepared {split_name} split is missing labels for {len(missing_split_labels)} images")
    split_counts[split_name] = len(split_image_paths)
print(f"Split counts: {split_counts}")
print("dataset.yaml:")
print(DATASET_YAML.read_text(encoding="utf-8"))

## 5. Configure the model and run

These values mirror the server benchmark configuration: YOLO26s pretrained weights, 100 maximum epochs, 640-pixel images, batch 24, AdamW, learning rate 0.001, deterministic seed 17, AMP disabled, and the same augmentation settings. Early stopping uses patience 20. Change `BATCH_SIZE` if this Colab GPU runs out of memory; for a clean comparison, keep all other settings fixed.

In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolo26s.pt"
PRETRAINED = True
IMAGE_SIZE = 640
BATCH_SIZE = 24
EPOCHS = 100
WORKERS = 4
DEVICE = 0
AMP = False
OPTIMIZER = "AdamW"
LEARNING_RATE = 0.001
PATIENCE = 20
FRACTION = 1.0

TRAIN_OPTIONS = {
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 15.0,
    "translate": 0.1,
    "scale": 0.5,
    "fliplr": 0.5,
    "flipud": 0.0,
    "mosaic": 1.0,
    "mixup": 0.1,
    "patience": PATIENCE,
}

run_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S_%f")
RUN_NAME = f"{run_timestamp}_colab_yolo26s"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)

training_config = {
    "mode": "benchmark",
    "model": MODEL_NAME,
    "pretrained": PRETRAINED,
    "dataset_yaml": str(DATASET_YAML),
    "data_root": str(PREPARED_DIR),
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "epochs": EPOCHS,
    "workers": WORKERS,
    "device": DEVICE,
    "amp": AMP,
    "optimizer": OPTIMIZER,
    "lr0": LEARNING_RATE,
    "seed": SEED,
    "deterministic": True,
    "fraction": FRACTION,
    "train_options": TRAIN_OPTIONS,
}
environment = {
    "python": __import__("sys").version,
    "platform": __import__("platform").platform(),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
}
(RUN_DIR / "config.json").write_text(json.dumps(training_config, indent=2) + "\n", encoding="utf-8")
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2) + "\n", encoding="utf-8")

print("Training configuration:")
print(json.dumps(training_config, indent=2))
print(f"Run artifacts will be saved to: {RUN_DIR}")
model = YOLO(MODEL_NAME)

## 6. Execute training

Ultralytics prints batch progress and per-epoch validation metrics here. The callbacks also collect epoch and validation durations, as the project training adapter does. Colab sessions can disconnect during long jobs, so keep checkpoints on Drive and avoid closing the tab while training is active.

In [ ]:
epoch_started = []
epoch_durations = []
validation_started = []
validation_durations = []

def on_epoch_start(trainer):
    epoch_started.append(time.perf_counter())

def on_epoch_end(trainer):
    if epoch_started:
        epoch_durations.append(time.perf_counter() - epoch_started[-1])

def on_validation_start(trainer):
    validation_started.append(time.perf_counter())

def on_validation_end(trainer):
    if validation_started:
        validation_durations.append(time.perf_counter() - validation_started.pop())

model.add_callback("on_train_epoch_start", on_epoch_start)
model.add_callback("on_train_epoch_end", on_epoch_end)
model.add_callback("on_val_start", on_validation_start)
model.add_callback("on_val_end", on_validation_end)

train_arguments = {
    "data": str(DATASET_YAML),
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "device": DEVICE,
    "amp": AMP,
    "workers": WORKERS,
    "optimizer": OPTIMIZER,
    "lr0": LEARNING_RATE,
    "seed": SEED,
    "deterministic": True,
    "fraction": FRACTION,
    "pretrained": PRETRAINED,
    "project": str(OUTPUT_ROOT),
    "name": RUN_NAME,
    "exist_ok": True,
    "plots": True,
    "verbose": True,
    "cache": False,
    "val": True,
    "save": True,
    "save_period": -1,
}
train_arguments.update(TRAIN_OPTIONS)

training_started_at = time.perf_counter()
train_result = model.train(**train_arguments)
total_training_seconds = time.perf_counter() - training_started_at
RUN_DIR = Path(getattr(train_result, "save_dir", RUN_DIR))
BEST_CHECKPOINT = RUN_DIR / "weights" / "best.pt"
LAST_CHECKPOINT = RUN_DIR / "weights" / "last.pt"

print(f"\nTraining call finished in {total_training_seconds / 60:.1f} minutes")
print(f"Epochs completed: {len(epoch_durations)}")
print(f"Mean epoch duration: {statistics.fmean(epoch_durations):.1f} seconds" if epoch_durations else "Epoch durations unavailable")
print(f"Validation time recorded by callbacks: {sum(validation_durations) / 60:.1f} minutes")
print(f"Best checkpoint: {BEST_CHECKPOINT} (exists={BEST_CHECKPOINT.is_file()})")
print(f"Last checkpoint: {LAST_CHECKPOINT} (exists={LAST_CHECKPOINT.is_file()})")
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Training finished without a best checkpoint at {BEST_CHECKPOINT}")
print("Ultralytics training metrics:")
print(getattr(train_result, "results_dict", {}))

## 7. Re-evaluate the best checkpoint

Training already validates each epoch. This explicit pass reloads `best.pt` and reports metrics on the prepared validation split. The dataset pipeline creates train and validation splits only, so this is validation evaluation, not an independent test-set score.

In [ ]:
best_model = YOLO(str(BEST_CHECKPOINT))
validation_result = best_model.val(
    data=str(DATASET_YAML),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=False,
    verbose=True,
)
validation_metrics = getattr(validation_result, "results_dict", {}) or {}
print("Validation metrics from best.pt:")
for metric_name, metric_value in validation_metrics.items():
    print(f"  {metric_name}: {metric_value}")

## 8. Run the `predict.py` inference flow

The repository's prediction script loads a checkpoint, predicts on an image or directory, prints each class/confidence/box, and saves annotated images. This cell performs the same single-image flow directly so the notebook is self-contained in Colab. By default it demonstrates inference on the first validation image; set `PREDICT_IMAGE` to another Drive image if desired.

In [ ]:
validation_image_dir = PREPARED_DIR / "images" / "val2017"
validation_image_paths = sorted(path for path in validation_image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
PREDICT_IMAGE = str(validation_image_paths[0])
PREDICTION_OUTPUT_DIR = RUN_DIR / "predictions"
PREDICTION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prediction_results = best_model.predict(
    source=PREDICT_IMAGE,
    device=DEVICE,
    batch=1,
    imgsz=IMAGE_SIZE,
    conf=0.25,
    iou=0.7,
    verbose=False,
)

for prediction in prediction_results:
    boxes = getattr(prediction, "boxes", None)
    if boxes is None or len(boxes) == 0:
        print(f"{prediction.path}: no detections")
    else:
        coordinates = boxes.xyxy.detach().cpu().tolist()
        class_ids = boxes.cls.detach().cpu().tolist()
        confidences = boxes.conf.detach().cpu().tolist()
        print(f"{prediction.path}: {len(coordinates)} detection(s)")
        for class_id, confidence, (x1, y1, x2, y2) in zip(class_ids, confidences, coordinates):
            label = prediction.names.get(int(class_id), str(int(class_id))) if isinstance(prediction.names, dict) else prediction.names[int(class_id)]
            print(f"  {label} confidence={confidence:.3f} xyxy=({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")
    output_image = PREDICTION_OUTPUT_DIR / Path(prediction.path).name
    prediction.save(filename=str(output_image))
    print(f"  saved: {output_image}")
    display(Image.open(output_image))

## 9. Save a compact run summary

The run directory in Drive contains Ultralytics' weights, CSV metrics, plots, and the configuration/environment records written above. This final cell adds JSON and Markdown summaries with elapsed time and validation accuracy.

In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if hasattr(value, "item"):
        try:
            return json_safe(value.item())
        except Exception:
            pass
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)

pure_training_seconds = sum(epoch_durations)
training_metrics = getattr(train_result, "results_dict", {}) or {}
run_summary = {
    "status": "completed",
    "model": MODEL_NAME,
    "checkpoint": str(BEST_CHECKPOINT),
    "run_directory": str(RUN_DIR),
    "dataset_yaml": str(DATASET_YAML),
    "dataset_size_images": len(image_paths),
    "split_counts": split_counts,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "epochs_requested": EPOCHS,
    "epochs_completed": len(epoch_durations),
    "amp": AMP,
    "optimizer": OPTIMIZER,
    "lr0": LEARNING_RATE,
    "total_training_seconds": total_training_seconds,
    "pure_train_seconds": pure_training_seconds,
    "validation_seconds": sum(validation_durations),
    "mean_epoch_seconds": statistics.fmean(epoch_durations) if epoch_durations else None,
    "epoch_times_seconds": epoch_durations,
    "training_accuracy": training_metrics,
    "validation_accuracy": validation_metrics,
    "environment": environment,
}
(RUN_DIR / "results.json").write_text(json.dumps(json_safe(run_summary), indent=2) + "\n", encoding="utf-8")

summary_lines = [
    "# MBDD2025 Colab training summary",
    "",
    f"- Status: {run_summary['status']}",
    f"- Model: {MODEL_NAME}",
    f"- Checkpoint: `{BEST_CHECKPOINT}`",
    f"- Images: {len(image_paths):,} total ({split_counts['train2017']:,} train, {split_counts['val2017']:,} validation)",
    f"- Configuration: imgsz={IMAGE_SIZE}, batch={BATCH_SIZE}, max epochs={EPOCHS}, AMP={AMP}, optimizer={OPTIMIZER}",
    f"- Epochs completed: {len(epoch_durations)}",
    f"- Training wall time: {total_training_seconds / 3600:.2f} hours",
    f"- Sum of epoch times: {pure_training_seconds / 3600:.2f} hours",
    "",
    "## Validation metrics",
]
summary_lines.extend(f"- {name}: {value}" for name, value in validation_metrics.items())
summary_lines.extend(["", f"- Prediction example: `{PREDICT_IMAGE}`", f"- Run directory: `{RUN_DIR}`", ""])
(RUN_DIR / "training_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Saved results.json and training_summary.md under {RUN_DIR}")
print(f"Best model checkpoint: {BEST_CHECKPOINT}")